In [3]:
%pip install ultralytics

  Using cached ultralytics_thop-2.0.19-py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   --------------------------------- ------ 1.0/1.3 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.3/1.3 MB 9.2 MB/s  0:00:00
   ---------------------------------------- 0.0/833.0 kB ? eta -:--:--
   ---------------------------------------- 833.0/833.0 kB 7.3 MB/s  0:00:00
   ---------------------------------------- 0.0/51.9 MB ? eta -:--:--
   -- ------------------------------------- 2.9/51.9 MB 15.2 MB/s eta 0:00:04
   ----- ---------------------------------- 7.1/51.9 MB 17.4 MB/s eta 0:00:03
   --------- ------------------------------ 12.1/51.9 MB 19.9 MB/s eta 0:00:03
   -------------- ------------------------- 19.1/51.9 MB 23.2 MB/s eta 0:00:02
   --------------------- ------------------ 27.5/51.9 MB 26.8 MB/s eta 0:00:01
   ----------------------------- ---------- 38.5/51.9 MB 31.4 MB/s eta 0:00:01
   ---------------


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\reino\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [ ]:
DATASET_ROOT = 'C:/Users/reino/Desktop/Учеба/Homework/Интеллектуальный анализ изображений/yolo-footprints'

In [ ]:
import os
import shutil
import yaml
import random
import glob
from ultralytics import YOLO

DATASET_ROOT = 'C:/Users/reino/Desktop/Учеба/Homework/Интеллектуальный анализ изображений/fhd_yolo-footprints'
SAVE_DIR = './lab14_results'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 150
DEVICE = 0
WORKERS = 0
CACHE_MODE = 'disk'


def split_dataset(root_dir):
    
    images_dir = os.path.join(root_dir, 'images')
    labels_dir = os.path.join(root_dir, 'labels')
    classes_file = os.path.join(root_dir, 'classes.txt')
    
    if not os.path.exists(images_dir) or not os.path.exists(labels_dir):
        raise FileNotFoundError(f"Папки images или labels не найдены в {root_dir}")

    img_extensions = ['*.jpg', '*.jpeg', '*.png', '*.bmp']
    all_images = []
    for ext in img_extensions:
        all_images.extend(glob.glob(os.path.join(images_dir, ext)))
    
    all_images.sort()
    random.seed(42)
    random.shuffle(all_images)
    
    n_total = len(all_images)
    n_valid = 20
    n_test = 20
    n_train = n_total - n_valid - n_test
    
    if n_train < 0:
        n_train = max(1, n_total - 2)
        n_valid = 1
        n_test = n_total - n_train - n_valid

    train_images = all_images[:n_train]
    valid_images = all_images[n_train:n_train + n_valid]
    test_images = all_images[n_train + n_valid:]
    
    print(f"Разделение: train={len(train_images)}, valid={len(valid_images)}, test={len(test_images)}")
    
    splits = {
        'train': train_images,
        'valid': valid_images,
        'test': test_images
    }
    
    dataset_path = os.path.join(SAVE_DIR, 'dataset')
    
    for split_name, images in splits.items():
        split_images_dir = os.path.join(dataset_path, split_name, 'images')
        split_labels_dir = os.path.join(dataset_path, split_name, 'labels')
        
        os.makedirs(split_images_dir, exist_ok=True)
        os.makedirs(split_labels_dir, exist_ok=True)
        
        for img_path in images:
            img_filename = os.path.basename(img_path)
            label_filename = os.path.splitext(img_filename)[0] + '.txt'
            
            shutil.copy(img_path, os.path.join(split_images_dir, img_filename))
            
            label_path = os.path.join(labels_dir, label_filename)
            if os.path.exists(label_path):
                shutil.copy(label_path, os.path.join(split_labels_dir, label_filename))
            else:
                with open(os.path.join(split_labels_dir, label_filename), 'w') as f:
                    pass
    
    if os.path.exists(classes_file):
        with open(classes_file, 'r') as f:
            classes = [line.strip() for line in f.readlines() if line.strip()]
    else:
        classes = ['object']
        print("classes.txt не найден, используется класс 'object' по умолчанию")
    
    yaml_data = {
        'path': os.path.abspath(dataset_path),
        'train': 'train',
        'val': 'valid',
        'test': 'test',
        'nc': len(classes),
        'names': classes
    }
    
    yaml_path = os.path.join(SAVE_DIR, 'data.yaml')
    with open(yaml_path, 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False)
    
    print(f"data.yaml создан: {yaml_path}")
    print(f"Классы: {classes}")
    
    return yaml_path


def run_task_3():
    
    try:
        yaml_path = split_dataset(DATASET_ROOT)
    except Exception as e:
        print(f"Ошибка при разделении датасета: {e}")
        return

    try:
        print("Загрузка модели YOLOv11n...")
        model = YOLO('https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt')
    except Exception as e:
        print(f"Не удалось загрузить модель: {e}")
        return

    print(f"Параметры: imgsz={IMG_SIZE}, batch={BATCH_SIZE}, epochs={EPOCHS}, workers={WORKERS}")
    
    experiment_name = 'task3_train'
    
    try:
        results = model.train(
            data=yaml_path,
            epochs=EPOCHS,
            batch=BATCH_SIZE,
            imgsz=IMG_SIZE,
            device=DEVICE,
            workers=WORKERS,
            cache=CACHE_MODE,
            rect=True,
            amp=True,
            save=True,
            project=SAVE_DIR,
            name=experiment_name
        )
    except RuntimeError as e:
        if "CUDA out of memory" in str(e) or "out of memory" in str(e).lower():
            print("Ошибка памяти")
        else:
            print(f"Ошибка обучения: {e}")
        return
    
    train_run_dir = os.path.join(SAVE_DIR, experiment_name)
    
    val_results = model.val(
        data=yaml_path,
        split='test',
        device=DEVICE,
        workers=0,
        imgsz=IMG_SIZE
    )
    
    print("\n" + "="*20 + " МЕТРИКИ " + "="*20)
    print(f"mAP50:     {val_results.box.map50:.4f}")
    print(f"mAP50-95:  {val_results.box.map:.4f}")
    print(f"Precision: {val_results.box.p.mean():.4f}")
    print(f"Recall:    {val_results.box.r.mean():.4f}")
    
    cm_src = os.path.join(train_run_dir, 'confusion_matrix.png')
    cm_dst = os.path.join(SAVE_DIR, 'task3_confusion_matrix.png')
    if os.path.exists(cm_src):
        shutil.copy(cm_src, cm_dst)
        print(f"\nConfusion matrix сохранена: {cm_dst}")
    else:
        print("\nConfusion matrix не найдена")
    
    # Путь к тестовым изображениям (исправлено)
    test_images_path = os.path.join(SAVE_DIR, 'dataset', 'test', 'images')
    
    print("\nГенерация предсказаний на тестовых изображениях...")
    print(f"Путь к тесту: {test_images_path}")
    
    if os.path.exists(test_images_path):
        model.predict(
            source=test_images_path,
            save=True,
            conf=0.20,
            iou=0.50,
            project=SAVE_DIR,
            name='task3_predictions',
            imgsz=IMG_SIZE,
            device=DEVICE
        )
    else:
        print(f"ВНИМАНИЕ: Папка {test_images_path} не найдена, пропускаем predict")
    
    print("\n" + "="*20 + " ЗАДАЧА 3 ВЫПОЛНЕНА " + "="*20)
    print(f"Результаты сохранены в: {SAVE_DIR}")
    
    return model, val_results

if __name__ == "__main__":
    model, metrics = run_task_3()

==================== ВЫПОЛНЕНИЕ ЗАДАЧИ 3 ====================
Разделение: train=200, valid=20, test=20
data.yaml создан: ./lab14_results\data.yaml
Классы: ['footprints']
Загрузка модели YOLOv11n...

Начало обучения...
Параметры: imgsz=640, batch=16, epochs=150, workers=0
Ultralytics 8.4.54  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./lab14_results\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, kera

Модель: YOLO11n

Обучение: 150 эпох, batch=16, imgsz=640, GPU RTX 4060

Итоговые метрики на тесте:
  - mAP50: 0.627
  - mAP50-95: 0.371  
  - Precision: 0.689
  - Recall: 0.642

In [14]:
import os
import cv2
import yaml
import random
import shutil
import numpy as np
from pathlib import Path
from ultralytics import YOLO


DATASET_ROOT = 'C:/Users/reino/Desktop/Учеба/Homework/Интеллектуальный анализ изображений/fhd_yolo-footprints'
SAVE_DIR = './lab14_results'
os.makedirs(SAVE_DIR, exist_ok=True)

IMG_SIZE = 640
BATCH_SIZE = 16
EPOCHS = 150
DEVICE = 0
WORKERS = 0
CACHE_MODE = 'disk'

AUG_FACTOR = 2
AUG_SEED = 42


def augment_yolo_dataset(train_dir, output_dir, factor=2, seed=42):
    """Создаёт аугментированные копии изображений с сохранением YOLO-разметки."""
    random.seed(seed)
    np.random.seed(seed)
    
    images_dir = Path(train_dir) / 'images'
    labels_dir = Path(train_dir) / 'labels'
    
    if not images_dir.exists() or not labels_dir.exists():
        raise FileNotFoundError(
            f"Директория датасета должна содержать подпапки images/ и labels/\n"
            f"Проверьте путь: {train_dir}"
        )
    
    out_images = Path(output_dir) / 'images'
    out_labels = Path(output_dir) / 'labels'
    
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)
    
    for img_path in images_dir.glob('*.*'):
        if img_path.suffix.lower() not in ['.jpg', '.jpeg', '.png', '.bmp']:
            continue
        shutil.copy(img_path, out_images / img_path.name)
        lbl_path = labels_dir / img_path.with_suffix('.txt').name
        if lbl_path.exists():
            shutil.copy(lbl_path, out_labels / lbl_path.name)
        else:
            (out_labels / lbl_path.name).touch()
    
    def apply_flip_h(img, boxes):
        boxes[:, 1] = 1.0 - boxes[:, 1]
        return cv2.flip(img, 1), boxes
    
    def apply_flip_v(img, boxes):
        boxes[:, 2] = 1.0 - boxes[:, 2]
        return cv2.flip(img, 0), boxes
    
    def apply_brightness(img, limit=30):
        delta = random.uniform(-limit, limit)
        return cv2.convertScaleAbs(img, alpha=1.0, beta=delta)
    
    def apply_noise(img, sigma=25):
        noise = np.random.normal(0, sigma, img.shape).astype(np.int16)
        return np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    def apply_blur(img, ksize=5):
        return cv2.GaussianBlur(img, (ksize, ksize), 0)
    
    img_files = list(images_dir.glob('*.jpg')) + list(images_dir.glob('*.png'))
    img_files = [f for f in img_files if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]
    
    processed = 0
    for img_path in img_files:
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        
        lbl_path = labels_dir / img_path.with_suffix('.txt').name
        if not lbl_path.exists():
            continue
        
        boxes = []
        with open(lbl_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) == 5:
                    boxes.append([float(p) for p in parts])
        
        if not boxes:
            continue
        
        boxes = np.array(boxes)
        
        for j in range(factor):
            aug_img = img.copy()
            aug_boxes = boxes.copy()
            
            if random.random() < 0.5:
                aug_img, aug_boxes = apply_flip_h(aug_img, aug_boxes)
            
            if random.random() < 0.3:
                aug_img, aug_boxes = apply_flip_v(aug_img, aug_boxes)
            
            if random.random() < 0.4:
                aug_img = apply_brightness(aug_img)
            
            if random.random() < 0.3:
                aug_img = apply_noise(aug_img)
            
            if random.random() < 0.2:
                aug_img = apply_blur(aug_img, ksize=random.choice([3, 5, 7]))
            
            new_name = f"{img_path.stem}_aug{j}{img_path.suffix}"
            cv2.imwrite(str(out_images / new_name), aug_img)
            
            with open(out_labels / f"{img_path.stem}_aug{j}.txt", 'w') as f:
                for box in aug_boxes:
                    f.write(f"{int(box[0])} {box[1]:.6f} {box[2]:.6f} {box[3]:.6f} {box[4]:.6f}\n")
            
            processed += 1
    
    print(f"Аугментация завершена: создано {processed} дополнительных изображений")
    print(f"Новый датасет: {output_dir}")
    return output_dir


def run_task_4():
    print("ВЫПОЛНЕНИЕ ЗАДАЧИ 4")
    
    original_train = os.path.join(SAVE_DIR, 'dataset', 'train')
    augmented_train = os.path.join(SAVE_DIR, 'dataset_aug', 'train')
    
    if not os.path.exists(original_train):
        raise FileNotFoundError(
            f"Исходная директория обучения не найдена: {original_train}\n"
            f"Убедитесь, что задача 3 выполнена и датасет подготовлен."
        )
    
    print("\n[1/4] Генерация аугментированных данных...")
    augment_yolo_dataset(
        train_dir=original_train,
        output_dir=augmented_train,
        factor=AUG_FACTOR,
        seed=AUG_SEED
    )
    
    print("\n[2/4] Обновление конфигурации датасета...")
    yaml_path = os.path.join(SAVE_DIR, 'data_aug.yaml')
    
    with open(os.path.join(SAVE_DIR, 'data.yaml'), 'r') as f:
        data_cfg = yaml.safe_load(f)
    
    data_cfg['train'] = os.path.abspath(augmented_train)
    
    with open(yaml_path, 'w') as f:
        yaml.dump(data_cfg, f, default_flow_style=False)
    
    print(f"Конфигурация сохранена: {yaml_path}")
    print(f"Train: {data_cfg['train']} (аугментированный)")
    print(f"Valid: {data_cfg['val']} (без изменений)")
    
    print("\n[3/4] Обучение модели на аугментированном датасете...")
    
    try:
        model = YOLO('https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt')
    except Exception as e:
        print(f"Не удалось загрузить модель: {e}")
        return None, None
    
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        imgsz=IMG_SIZE,
        device=DEVICE,
        workers=WORKERS,
        cache=CACHE_MODE,
        rect=True,
        amp=True,
        save=True,
        project=SAVE_DIR,
        name='task4_train_aug'
    )
    
    print("\n[4/4] Тестирование модели...")
    val_results = model.val(
        data=yaml_path,
        split='test',
        device=DEVICE,
        workers=0,
        imgsz=IMG_SIZE
    )
    
    print("\nМЕТРИКИ ЗАДАЧИ 4 (с аугментацией)")
    print(f"mAP50:     {val_results.box.map50:.4f}")
    print(f"mAP50-95:  {val_results.box.map:.4f}")
    print(f"Precision: {val_results.box.p.mean():.4f}")
    print(f"Recall:    {val_results.box.r.mean():.4f}")
    
    test_images_path = os.path.join(SAVE_DIR, 'dataset', 'test', 'images')
    if os.path.exists(test_images_path):
        model.predict(
            source=test_images_path,
            save=True,
            conf=0.25,
            project=SAVE_DIR,
            name='task4_predictions_aug',
            imgsz=IMG_SIZE,
            device=DEVICE
        )
    
    return model, val_results


def compare_results(metrics_task3, metrics_task4):
    """Выводит сравнительную таблицу метрик."""
    print("\nСРАВНЕНИЕ РЕЗУЛЬТАТОВ")
    print(f"{'Метрика':<12} | {'Задача 3':<10} | {'Задача 4':<10} | {'Изменение':<10}")
    print("-" * 60)
    
    metrics = [
        ('mAP50', metrics_task3.box.map50, metrics_task4.box.map50),
        ('mAP50-95', metrics_task3.box.map, metrics_task4.box.map),
        ('Precision', metrics_task3.box.p.mean(), metrics_task4.box.p.mean()),
        ('Recall', metrics_task3.box.r.mean(), metrics_task4.box.r.mean()),
    ]
    
    for name, v3, v4 in metrics:
        delta = v4 - v3
        sign = '+' if delta >= 0 else ''
        print(f"{name:<12} | {v3:<10.4f} | {v4:<10.4f} | {sign}{delta:<9.4f}")
    
    print("-" * 60)


if __name__ == "__main__":
    class MockResults:
        class Box:
            def __init__(self, map50, map, p, r):
                self.map50 = map50
                self.map = map
                self.p = np.array([p])
                self.r = np.array([r])
                self.mean = lambda: self.p[0] if hasattr(self.p, '__getitem__') else self.p
        
        def __init__(self, map50, map, p, r):
            self.box = self.Box(map50, map, p, r)
    
    metrics_task3 = MockResults(
        map50=0.6266,
        map=0.3711,
        p=0.6886,
        r=0.6415
    )
    
    model_aug, metrics_task4 = run_task_4()
    
    if metrics_task4 is not None:
        compare_results(metrics_task3, metrics_task4)
        print(f"\nРезультаты сохранены в: {SAVE_DIR}")

ВЫПОЛНЕНИЕ ЗАДАЧИ 4

[1/4] Генерация аугментированных данных...
Аугментация завершена: создано 400 дополнительных изображений
Новый датасет: ./lab14_results\dataset_aug\train

[2/4] Обновление конфигурации датасета...
Конфигурация сохранена: ./lab14_results\data_aug.yaml
Train: c:\Users\reino\Desktop\Учеба\Homework\Интеллектуальный анализ изображений\lab14_results\dataset_aug\train (аугментированный)
Valid: valid (без изменений)

[3/4] Обучение модели на аугментированном датасете...
Found https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11n.pt locally at weights\yolo11n.pt
Ultralytics 8.4.54  Python-3.11.9 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./lab14_resul

In [ ]:
СРАВНЕНИЕ РЕЗУЛЬТАТОВ
Метрика      | Задача 3   | Задача 4   | Изменение 
------------------------------------------------------------
mAP50        | 0.6266     | 0.5736     | -0.0530  
mAP50-95     | 0.3711     | 0.3566     | -0.0145  
Precision    | 0.6886     | 0.6283     | -0.0603  
Recall       | 0.6415     | 0.6038     | -0.0377  
------------------------------------------------------------